In [1]:
import os
os.chdir("D:/Projects/volatility-radar")

In [2]:
import pandas as pd

df = pd.read_csv("data/final_cleaned_data/full_features.csv")
df.shape

(4023, 40)

In [3]:
df['date'] = pd.to_datetime(df['date'])
unique_dates = df['date'].unique()

print(unique_dates[0])
print(unique_dates[-1])

2021-01-21 00:00:00
2026-03-12 00:00:00


In [4]:
def walk_forward_splits(unique_dates, initial_train_months=24, test_months=3, gap_days=14):
    splits = []
    unique_dates = pd.Series(unique_dates).sort_values().reset_index(drop=True)
    
    train_start = unique_dates.iloc[0]
    first_test_start = train_start + pd.DateOffset(months=initial_train_months)
    
    test_start = first_test_start
    
    while True:
        test_end = test_start + pd.DateOffset(months=test_months)
        
        # stop if test window exceeds available data
        if test_end > unique_dates.iloc[-1]:
            break
        
        train_end = test_start - pd.Timedelta(days=gap_days)
        
        train_dates = unique_dates[unique_dates <= train_end]
        test_dates = unique_dates[(unique_dates >= test_start) & (unique_dates <= test_end)]
        
        # only add fold if both windows have data
        if len(train_dates) > 0 and len(test_dates) > 0:
            splits.append((train_dates.values, test_dates.values))
        
        test_start = test_start + pd.DateOffset(months=test_months)
    
    return splits

In [5]:
# def walk_forward_splits_1(unique_dates, months = 24, gap = 21, test_days = 14):
#     spilts = []

#     unique_dates = pd.Series(unique_dates).sort_values().reset_index(drop= True)

#     train_start_date = unique_dates.iloc[0]

#     test_start_date = unique_dates.iloc[0] + pd.DateOffset(months) + pd.DateOffset(gap)

#     while start_date < unique_dates[-1]:




In [6]:
splits = walk_forward_splits(unique_dates)

print(f"Total folds: {len(splits)}")
for i, (train, test) in enumerate(splits):
    print(f"Fold {i+1}: Train {train[0].astype('datetime64[D]')} → {train[-1].astype('datetime64[D]')} ({len(train)} days) | Test {test[0].astype('datetime64[D]')} → {test[-1].astype('datetime64[D]')} ({len(test)} days)")

Total folds: 12
Fold 1: Train 2021-01-21 → 2023-01-06 (512 days) | Test 2023-01-23 → 2023-04-21 (65 days)
Fold 2: Train 2021-01-21 → 2023-04-07 (577 days) | Test 2023-04-21 → 2023-07-21 (66 days)
Fold 3: Train 2021-01-21 → 2023-07-07 (642 days) | Test 2023-07-21 → 2023-10-20 (66 days)
Fold 4: Train 2021-01-21 → 2023-10-06 (707 days) | Test 2023-10-23 → 2024-01-19 (65 days)
Fold 5: Train 2021-01-21 → 2024-01-05 (772 days) | Test 2024-01-22 → 2024-04-19 (65 days)
Fold 6: Train 2021-01-21 → 2024-04-05 (837 days) | Test 2024-04-22 → 2024-07-19 (65 days)
Fold 7: Train 2021-01-21 → 2024-07-05 (902 days) | Test 2024-07-22 → 2024-10-21 (66 days)
Fold 8: Train 2021-01-21 → 2024-10-07 (968 days) | Test 2024-10-21 → 2025-01-21 (67 days)
Fold 9: Train 2021-01-21 → 2025-01-07 (1034 days) | Test 2025-01-21 → 2025-04-21 (65 days)
Fold 10: Train 2021-01-21 → 2025-04-07 (1098 days) | Test 2025-04-21 → 2025-07-21 (66 days)
Fold 11: Train 2021-01-21 → 2025-07-07 (1163 days) | Test 2025-07-21 → 2025-10-21

In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
import numpy as np

FEATURE_COLS = [
    'pair', 'day_of_week', 'month', 'week_of_year', 'is_month_end', 'is_month_start',
    'max_up_pips', 'max_down_pips', 'max_profit', 'max_loss', 'daily_return',
    'return_3d', 'return_5d', 'return_10d', 'rolling_std_5', 'rolling_std_10',
    'rolling_std_20', 'rsi_14', 'atr_14', 'momentum_5d', 'momentum_10d',
    'dist_from_mean_20d', 'daily_range', 'candle_body', 'upper_wick', 'lower_wick',
    'high_impact_count', 'medium_impact_count', 'low_impact_count', 'max_z_score',
    'sum_signal', 'dominant_direction', 'max_surprise_z', 'sum_signal_surprise'
]

TARGET_COL = 'label'

def run_walk_forward(df, splits):
    fold_results = []

    for i, (train_dates, test_dates) in enumerate(splits):
        train_df = df[df['date'].isin(train_dates)].copy()
        test_df = df[df['date'].isin(test_dates)].copy()

        X_train = train_df[FEATURE_COLS]
        y_train = train_df[TARGET_COL]
        X_test = test_df[FEATURE_COLS]
        y_test = test_df[TARGET_COL]

        model = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        weighted_f1 = f1_score(y_test, y_pred, average='weighted')
        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

        fold_results.append({
            'fold': i + 1,
            'train_size': len(train_df),
            'test_size': len(test_df),
            'weighted_f1': weighted_f1,
            'report': report,
            'y_test': y_test.values,
            'y_pred': y_pred
        })

        print(f"Fold {i+1} | Train: {len(train_df)} rows | Test: {len(test_df)} rows | Weighted F1: {weighted_f1:.4f}")

    return fold_results

fold_results = run_walk_forward(df, splits)

Fold 1 | Train: 1536 rows | Test: 195 rows | Weighted F1: 0.3950
Fold 2 | Train: 1731 rows | Test: 198 rows | Weighted F1: 0.3631
Fold 3 | Train: 1926 rows | Test: 198 rows | Weighted F1: 0.3651
Fold 4 | Train: 2121 rows | Test: 195 rows | Weighted F1: 0.3284
Fold 5 | Train: 2316 rows | Test: 195 rows | Weighted F1: 0.3211
Fold 6 | Train: 2511 rows | Test: 195 rows | Weighted F1: 0.3937
Fold 7 | Train: 2706 rows | Test: 198 rows | Weighted F1: 0.3140
Fold 8 | Train: 2904 rows | Test: 201 rows | Weighted F1: 0.3544
Fold 9 | Train: 3102 rows | Test: 195 rows | Weighted F1: 0.3646
Fold 10 | Train: 3294 rows | Test: 198 rows | Weighted F1: 0.3496
Fold 11 | Train: 3489 rows | Test: 201 rows | Weighted F1: 0.3139
Fold 12 | Train: 3687 rows | Test: 201 rows | Weighted F1: 0.2953


In [8]:
from sklearn.metrics import confusion_matrix
import numpy as np

all_y_test = np.concatenate([r['y_test'] for r in fold_results])
all_y_pred = np.concatenate([r['y_pred'] for r in fold_results])

print("Aggregated confusion matrix (all 12 folds):")
print(confusion_matrix(all_y_test, all_y_pred))
print("\nRows = actual, Columns = predicted")
print("Classes: 0=Bearish, 1=Neutral, 2=Bullish")

from sklearn.metrics import classification_report
print(classification_report(all_y_test, all_y_pred, target_names=['Bearish','Neutral','Bullish']))

Aggregated confusion matrix (all 12 folds):
[[339 168 282]
 [271 142 284]
 [326 189 369]]

Rows = actual, Columns = predicted
Classes: 0=Bearish, 1=Neutral, 2=Bullish
              precision    recall  f1-score   support

     Bearish       0.36      0.43      0.39       789
     Neutral       0.28      0.20      0.24       697
     Bullish       0.39      0.42      0.41       884

    accuracy                           0.36      2370
   macro avg       0.35      0.35      0.35      2370
weighted avg       0.35      0.36      0.35      2370



In [9]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'eval_metric': 'mlogloss',
        'random_state': 42,
        'n_jobs': -1
    }

    # use only first 6 folds for tuning — saves time
    fold_f1s = []
    for i, (train_dates, test_dates) in enumerate(splits[:6]):
        train_df = df[df['date'].isin(train_dates)]
        test_df = df[df['date'].isin(test_dates)]

        X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
        X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET_COL]

        model = XGBClassifier(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        fold_f1s.append(f1_score(y_test, y_pred, average='weighted'))

    return np.mean(fold_f1s)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest weighted F1: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

  0%|          | 0/50 [00:00<?, ?it/s]


Best weighted F1: 0.3746
Best params: {'n_estimators': 477, 'max_depth': 6, 'learning_rate': 0.17117589219455565, 'subsample': 0.8320514307388407, 'colsample_bytree': 0.8651041698597045, 'min_child_weight': 2, 'gamma': 1.7940824336390082}


In [10]:
print(df['label'].value_counts(normalize=True).sort_index())

label
0.0    0.334328
1.0    0.297539
2.0    0.368133
Name: proportion, dtype: float64


In [ ]:
PAIRS = {0: 'EURUSD', 1: 'GBPUSD', 2: 'USDJPY'}

per_pair_results = {}

for pair_id, pair_name in PAIRS.items():
    print(f"\n--- {pair_name} ---")
    pair_df = df[df['pair'] == pair_id].copy()
    
    pair_unique_dates = pair_df['date'].sort_values().unique()
    pair_splits = walk_forward_splits(pair_unique_dates)
    
    # drop pair column — not needed when training per pair
    pair_feature_cols = [c for c in FEATURE_COLS if c != 'pair']
    
    fold_f1s = []
    fold_preds = []
    
    for i, (train_dates, test_dates) in enumerate(pair_splits):
        train_df = pair_df[pair_df['date'].isin(train_dates)]
        test_df = pair_df[pair_df['date'].isin(test_dates)]
        
        X_train, y_train = train_df[pair_feature_cols], train_df[TARGET_COL]
        X_test, y_test = test_df[pair_feature_cols], test_df[TARGET_COL]
        
        model = XGBClassifier(
            n_estimators=482,
            max_depth=3,
            learning_rate=0.11273329850596055,
            subsample=0.66215706436737,
            colsample_bytree=0.9022585181959433,
            min_child_weight=4,
            gamma=2.1083773586540033,
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=-1
        )
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        wf1 = f1_score(y_test, y_pred, average='weighted')
        fold_f1s.append(wf1)
        fold_preds.append((y_test.values, y_pred))
        
        print(f"Fold {i+1} | F1: {wf1:.4f}")
    
    avg_f1 = np.mean(fold_f1s)
    print(f"{pair_name} Average F1: {avg_f1:.4f}")
    per_pair_results[pair_name] = {'fold_f1s': fold_f1s, 'avg_f1': avg_f1, 'fold_preds': fold_preds}

In [ ]:
import joblib
import os

final_model = XGBClassifier(
    n_estimators=482,
    max_depth=3,
    learning_rate=0.11273329850596055,
    subsample=0.66215706436737,
    colsample_bytree=0.9022585181959433,
    min_child_weight=4,
    gamma=2.1083773586540033,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

X_full = df[FEATURE_COLS]
y_full = df[TARGET_COL]

final_model.fit(X_full, y_full)

os.makedirs('src/models', exist_ok=True)
joblib.dump(final_model, 'src/models/volatility_radar_xgb.pkl')
print("Model saved.")

In [11]:
from dotenv import load_dotenv
load_dotenv(dotenv_path='D:/Projects/volatility-radar/.env')

True

In [13]:
from openai import OpenAI
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print("KEY:", api_key[:8] if api_key else "NONE")  # prints first 8 chars only
client = OpenAI(api_key=api_key)

KEY: NONE


In [14]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path='D:/Projects/volatility-radar/.env')

True